`Dataset`: images of triangles labeled 1 and 0, with variations in size, position, rotation, color, noise, random lines, and background artifacts.

# 0. Test Objective

This notebook is a mandatory recruitment exercise. The goal is to evaluate whether the candidate can build a rigorous end-to-end computer vision pipeline, not only maximize a score.

The candidate is evaluated on:
- data understanding and visual diagnosis,
- leakage-free dataset splitting and reproducibility,
- low-level implementation of metrics and decision thresholds,
- baseline quality and honest comparison against stronger models,
- representation analysis with clustering and PCA,
- supervised modeling with dummy, logistic regression, XGBoost, CNN embeddings, CNN + MLP, and generative augmentation,
- calibration, robustness, OOD behavior, and error analysis,
- code quality, execution reliability, and clarity of conclusions.

All sections are mandatory. When a method performs poorly, the candidate must explain why and keep the result in the comparison table.

## 0.1 Mandatory No-Pythagorean-Theorem Rule

The candidate must not explicitly state, implement, encode, approximate, or use the Pythagorean theorem in any way.

Forbidden shortcuts include, but are not limited to:
- checking whether `a^2 + b^2 = c^2`;
- computing side lengths and applying a hand-coded right-triangle rule;
- detecting right angles with an explicit geometric formula;
- creating labels, features, metrics, filters, or post-processing rules based on the Pythagorean theorem.

The task must be solved through machine learning, representation learning, robustness analysis, and model interpretation. If geometric quantities are used for diagnostics, they must not implement or reveal a deterministic Pythagorean decision rule.

## 0.2 Execution Instructions

Expected input:
- `gen-images.py` must be available at the project root.
- Run `python gen-images.py` from the project root to generate `triangle_images/0` and `triangle_images/1`.
- The generator uses `N_SAMPLES = 20000`, `RECTANGLE_PCT = 0.3`, `IMAGE_SIZE = 256`, and `SEED = 42`.
- Class `1` represents right triangles. Class `0` represents non-right triangles.
- Do not manually move images after generation. Use the generated paths and save the split membership explicitly.

Environment:
- Python 3.10+.
- Suggested dependencies: `numpy`, `pandas`, `matplotlib`, `seaborn`, `pillow`, `scikit-learn`, `xgboost`, `hdbscan`, `optuna`, `shap`, `torch`, `torchvision`, `tqdm`.
- CPU is sufficient for EDA, PCA, clustering, logistic regression, XGBoost, and metric implementation.
- GPU is recommended for CNN, CNN embeddings, and GAN training. If only CPU is available, training can be shorter, but the candidate must document runtime constraints and keep model comparisons fair.

Reproducibility:
- Use seed `42` for `random`, `numpy`, `scikit-learn`, `torch`, train/validation splits, and model initialization whenever possible.
- Save split indices or image paths before training any model.
- Report the exact preprocessing used for each model.

Expected effort:
- Full exercise: roughly one working day.
- Notebook execution target: under 2 hours on a recent GPU. CPU-only execution can be longer and must be reported.

## 0.3 Expected Deliverables

The candidate must submit:
- the completed notebook with all cells executed and outputs visible,
- a short final report, either as a final notebook section & a `report.md` / `report.pdf`, limited to 2 pages,
- a model comparison table saved as `results/model_comparison.csv`,
- final predictions saved as `results/final_predictions.csv` with at least: `model`, `split`, `image_path`, `y_true`, `y_proba`, `y_pred`, `threshold`,
- split membership saved as `results/split_indices.json` or `results/split_indices.csv`,
- the final selected model configuration and, when applicable, saved weights or serialized model artifacts,
- the two manually drawn final test images and their predictions,
- environment notes: Python version, package versions, CPU/GPU information, total runtime.

The final comparison table must include at least: `model`, `features`, `threshold_strategy`, `accuracy`, `balanced_accuracy`, `precision`, `recall`, `specificity`, `f1`, `mcc`, `roc_auc`, `log_loss`, `brier_score`, `predicted_positive_rate`, `prevalence`, `calibration_method`, `fit_time_seconds`, `inference_time_seconds`, `D_test_score`, `D_guard_score`, and `robustness_gap`.

The final report must include: best model choice, reasons for rejecting weaker models, main failure modes, evidence of robustness or lack of robustness, and what would be improved with more time.

## 0.4 Mandatory Non-Regression Checks

Before modeling, implement and run simple assert-based checks:
- the generated dataset contains exactly `20000` images when the default generator is used,
- class distribution is exactly `14000` images for class `0` and `6000` images for class `1`,
- every image belongs to exactly one split among `D_world`, `D_dev`, `D_calib`, `D_test`, and `D_guard`,
- there is no path, filename, augmented duplicate, or transformed copy leakage across splits,
- raw images and tensors have consistent shapes before entering each model,
- `y_true`, `y_pred`, and `y_proba` have matching lengths,
- probabilities are finite values in `[0, 1]`,
- binary predictions contain only `0` and `1`,
- metric values are finite and remain in their valid ranges,
- the dummy majority baseline accuracy matches the majority class prevalence on each evaluation split,
- every final model is evaluated on the same `D_test` and `D_guard` samples.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
DATA_DIR = Path("triangle_images")
EXPECTED_N = 20_000
EXPECTED_COUNTS = {0: 14_000, 1: 6_000}
SPLIT_NAMES = {"D_world", "D_dev", "D_calib", "D_test", "D_guard"}

def assert_dataset_integrity(data_dir=DATA_DIR):
    counts = {
        label: len(list((data_dir / str(label)).glob("*.png")))
        for label in EXPECTED_COUNTS
    }
    assert sum(counts.values()) == EXPECTED_N, f"Expected {EXPECTED_N} images, got {counts}"
    assert counts == EXPECTED_COUNTS, f"Unexpected class distribution: {counts}"
    return counts

def assert_no_split_leakage(split_to_paths):
    assert set(split_to_paths) == SPLIT_NAMES, f"Unexpected split names: {set(split_to_paths)}"
    seen = {}
    for split_name, paths in split_to_paths.items():
        normalized = [str(Path(path)) for path in paths]
        assert len(normalized) == len(set(normalized)), f"Duplicate path inside {split_name}"
        overlap = set(normalized).intersection(seen)
        assert not overlap, f"Leakage detected in {split_name}: {list(overlap)[:5]}"
        seen.update({path: split_name for path in normalized})
    return True

def assert_binary_outputs(y_true, y_pred, y_proba):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_proba = np.asarray(y_proba, dtype=float)
    assert y_true.shape == y_pred.shape == y_proba.shape
    assert set(np.unique(y_true)).issubset({0, 1})
    assert set(np.unique(y_pred)).issubset({0, 1})
    assert np.isfinite(y_proba).all()
    assert ((0.0 <= y_proba) & (y_proba <= 1.0)).all()
    return True

def assert_dummy_majority_baseline(y_true, dummy_pred):
    y_true = np.asarray(y_true).astype(int)
    dummy_pred = np.asarray(dummy_pred).astype(int)
    majority_class = np.bincount(y_true).argmax()
    expected_accuracy = np.mean(y_true == majority_class)
    observed_accuracy = np.mean(y_true == dummy_pred)
    assert np.isclose(observed_accuracy, expected_accuracy), (
        f"Dummy accuracy mismatch: expected {expected_accuracy:.4f}, got {observed_accuracy:.4f}"
    )
    return expected_accuracy

def assert_metric_table(metrics_df):
    required_columns = {
        "model", "split", "accuracy", "balanced_accuracy", "precision", "recall",
        "specificity", "f1", "mcc", "roc_auc", "log_loss", "brier_score"
    }
    missing = required_columns.difference(metrics_df.columns)
    assert not missing, f"Missing metric columns: {sorted(missing)}"
    numeric_cols = list(required_columns.difference({"model", "split"}))
    assert np.isfinite(metrics_df[numeric_cols].to_numpy(dtype=float)).all()
    bounded_cols = ["accuracy", "balanced_accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "brier_score"]
    for col in bounded_cols:
        assert ((0.0 <= metrics_df[col]) & (metrics_df[col] <= 1.0)).all(), f"{col} out of range"
    return True


# 1. Data Analysis

1. How many images are contained in the dataset?
2. What is the class distribution?
3. Is the dataset balanced?
4. Display 10 images from each class.
5. Which features are most informative?

## 2. Clustering

Goal: analyze whether images naturally group together without labels.

Questions
1. What happens when applying K-Means on raw pixels?
2. Do the clusters correspond to the labels?
3. Are clusters driven more by color, noise, or shape?
4. Why can unsupervised clustering fail on this dataset?
5. Compare K-Means with HDBSCAN.
6. Visualize representative images from each cluster.
7. Are clustering mistakes related to the background?

Mandatory visualizations:
- For every clustering method used, display the clustering graph.
- For K-Means, show at least one 2D projection colored by cluster and another colored by true label.
- For HDBSCAN, show the same type of graph and make noise points explicit with a distinct color or marker.
- If clustering is run on several feature spaces, for example raw pixels, PCA features, or CNN embeddings, display a clustering graph for each feature space/method pair.
- Include a short interpretation below each graph explaining whether the clusters reflect shape, color, noise, position, or background artifacts.

## 3. PCA


Goal: reduce dimensionality and analyze latent structure.

Questions
1. How much variance is explained by the first principal components?
2. Are the two classes separable in 2D PCA space?
3. What do the first principal components visually represent?
4. Does PCA capture shape or background artifacts?
5. Can a classifier be trained only on PCA components?
6. Compare XGBoost on raw pixels vs PCA features.
7. Does PCA improve or degrade performance?

## 4. Define Model Evaluation

`Note`: implement these metrics at a low level. For example, classification metrics should only have access to: `y_pred`, `y_proba`, and `y_true`.

Main questions the metrics should help answer:
- Does the model generalize or memorize?
- Does the model rely on background artifacts?
- Which architecture is the most robust?
- Which architecture is the most stable?
- Which architecture is the most interpretable?
- Which architecture produces the best latent structure?
- Which architecture handles OOD best?
- Which model is the most computationally efficient?
- Is a generative model useful for augmentation?
- Does adding complexity actually improve generalization?

Goal: define a rigorous evaluation framework to measure:

- classification quality,
- robustness,
- stability,
- generalization,
- decision quality,
- latent structure quality,
- cooperation/supermodularity between models,
- OOD resistance.

### 4.1 Classification Metrics

Core supervised metrics.

Metrics:
- Accuracy
- Precision
- Recall
- Specificity
- F1-score
- Balanced Accuracy
- Matthews Correlation Coefficient (MCC)
- ROC-AUC
- Log Loss
- Brier Score
- Predicted Positive Rate
- Prevalence

### 4.2 Decision Metrics

Decision-space analysis based on confidence and threshold behavior.

Metrics:
- Decision confidence
- Threshold distance
- Threshold confidence ratio
- Threshold confidence gap
- Threshold drift
- Implicit linear threshold
- Top threshold candidates

### 4.3 Stability Metrics

Measure fold-to-fold consistency and robustness.

Metrics
- AUC standard deviation
- LogLoss standard deviation
- Recall standard deviation
- Precision standard deviation
- Predicted Positive Rate standard deviation
- Threshold standard deviation
- Threshold drift
- Robustness gap
- Stability score

### 4.4 Generalization Metrics 

Analyze whether the model truly generalizes or memorizes.

Metrics:
- bias
- variance
- residual noise
- overfitting
- Worst fold performance
- Robustness gap
- Interfold variance
- Interfold standard deviation

### 4.5 CNN Generalization Metrics

Specific metrics for neural network training dynamics.

Metrics
- Best validation loss
- Best validation epoch
- Final overfitting gap
- Overfitting score
- Overfitting epoch
- Validation loss degradation
- Epochs after best validation
- Overfitting area
- Validation loss increase count
- Validation loss increase rate

### 4.6 Cooperation / Supermodularity Metrics

Analyze whether combining models improves performance.

Example:

`CNN embeddings + XGB`

vs

`CNN alone`

Metrics:
- Supermodularity gain
- Interaction score
- Combined AUC
- Combined MCC
- Combined F1
- Combined LogLoss

```
Examples of questions:
Does combining CNN + XGB improve performance?
Is the gain larger than expected?
Are embeddings useful for XGB?
Which architecture combination is most synergistic?
Does the combined system generalize better?
```

## 5. Dataset Split

You should split the dataset into:
- `D_world`: Dataset used for representation learning, latent learning, generative models, and self-supervised learning.
- `D_dev`: Dataset used for supervised training, model development, architecture experimentation, and hyperparameter optimization.
- `D_calib`: Dataset used for probability calibration, threshold selection, and decision calibration.
- `D_test`: Dataset used for final in-distribution evaluation and model comparison.
- `D_guard`: Dataset used for robustness testing, stress testing, OOD evaluation, and generalization analysis.

## 6. Simple Baselines

Goal: establish minimum reference levels before building complex models.

Mandatory baselines:
- `DummyClassifier`: majority-class prediction and, optionally, stratified random prediction.
- `LogisticRegression`: trained on flattened raw pixels after a documented preprocessing step.
- `XGBoost raw pixels`: trained on flattened raw pixels using the same split as all other models.

Questions
1. What is the score of a model that only predicts the majority class?
2. Does logistic regression learn anything beyond class prevalence?
3. Does XGBoost on raw pixels outperform dummy and logistic baselines?
4. Which baseline is most sensitive to color, position, rotation, and background artifacts?
5. How much improvement is required before a CNN is justified?

Deliverables
- Confusion matrix and all mandatory metrics for each baseline.
- Fit time and inference time for each baseline.
- Short interpretation explaining what each baseline proves or fails to prove.

## 7. XGBoost Baseline

Goal: create a classical ML baseline.

Questions
1. How can images be converted into vectors?
2. Why can XGBoost work even without CNNs?
3. What are the limitations of XGBoost on raw pixels?
4. Is XGBoost sensitive to triangle position?
5. Is it sensitive to color?
6. Compare XGBoost on:
- raw pixels,
- PCA features,
- CNN embeddings.

## 8. CNN + XGBoost

`pipeline`:
```
Image → CNN Encoder → Embedding → XGBoost Classifier
```

Questions
- Why use a CNN as a feature extractor?
- Why place XGBoost after the CNN?
- Do CNN embeddings separate the classes better?
- Compare against XGBoost on raw pixels.
- Visualize embeddings using PCA or t-SNE.
- Does XGBoost exploit learned features more effectively?

## 9. CNN + MLP

`Pipeline`
```
Image → CNN → MLP → Prediction
```

Questions
- What is the difference compared to CNN + XGBoost?
- Does the MLP outperform XGBoost?
- Is the model overfitting?
- Compare training loss and validation loss.
- Does dropout improve generalization?
- Does data augmentation improve robustness?

## 10. Generator + Discriminator


Goal: introduce generative modeling.

Questions
- Can the model generate realistic new triangles?
- Does the discriminator learn meaningful differences?
- Can generated images augment the dataset?
- Does generative augmentation improve CNN performance?
- Does the generator reproduce shape or mostly noise?
- How should generated image quality be evaluated?

```
Train GAN → Generate New Images → Add to Training Set → Compare Performance
```

## 11. World Model / Self-Supervised Representation Learning

Goal: learn the structure of the triangle image world without using labels, then test whether the learned representation improves classification, robustness, and OOD detection.

Mandatory world model work:
- Train the world model only on `D_world`.
- Do not use labels during world model training.
- Choose at least one self-supervised or generative representation method: autoencoder, variational autoencoder, masked autoencoder, contrastive learning / SimCLR, or another justified approach.
- Extract embeddings `z = encoder(image)` for `D_dev`, `D_calib`, `D_test`, and `D_guard`.
- Train at least one downstream classifier on world-model embeddings, for example Logistic Regression, XGBoost, or MLP.
- Compare world-model embeddings against raw pixels, PCA features, and CNN embeddings.
- Use reconstruction error or latent-distance scores as an OOD signal on `D_guard`.
- Visualize the latent space with PCA or UMAP, colored by true label and by relevant artifact metadata when available.

Mandatory visualizations:
- Original images vs reconstructed images.
- Latent-space projection of world-model embeddings.
- Interpolation between two latent vectors, with decoded images if the model has a decoder.
- Examples with high reconstruction error and low reconstruction error.

Metrics:
- Reconstruction MAE or MSE.
- SSIM if implemented.
- Downstream classification metrics on embeddings.
- Latent clustering purity or separation quality.
- OOD AUROC or another justified OOD score comparing `D_test` and `D_guard`.
- Invariance violation rate: how often a label-preserving transformation changes the final classifier prediction.

Questions
1. Does the world model learn triangle geometry or mostly background artifacts?
2. Are world-model embeddings more robust than raw pixels and PCA features?
3. Does reconstruction error detect difficult or OOD examples?
4. Do downstream models trained on world-model embeddings improve generalization?
5. What does latent interpolation reveal about the learned representation?

## 12. Calibration, Thresholds, and Final Metrics

Goal: turn model scores into reliable decisions and compare models under a shared decision framework.

Mandatory calibration and threshold work:
- Use `D_calib` only for probability calibration and threshold selection.
- Compare at least one uncalibrated model with calibrated variants, for example Platt scaling / sigmoid calibration and isotonic calibration.
- Select thresholds using explicit criteria: best F1, best MCC, target recall, balanced accuracy, or cost-sensitive objective.
- Report how the selected threshold changes predicted positive rate, recall, precision, specificity, MCC, log loss, and Brier score.
- Apply the selected threshold once to `D_test` and `D_guard`; do not tune on these splits.
- Plot calibration curves / reliability diagrams for the main models.

Questions
1. Are model probabilities trustworthy before calibration?
2. Which calibration method improves Brier score and log loss?
3. Which thresholding strategy gives the best decision trade-off?
4. Does the threshold selected on `D_calib` remain stable on `D_test` and `D_guard`?
5. Do calibrated probabilities improve robustness or only decision interpretability?

## 13. Evaluation

Goal: compare all models and model combinations under the same protocol, using the same splits, metrics, calibration rules, and robustness tests.

Evaluation protocol:
- Use `D_dev` for model fitting and architecture iteration.
- Use `D_calib` only for probability calibration, threshold selection, and ensemble weight selection.
- Use `D_test` only for final in-distribution evaluation.
- Use `D_guard` only for robustness, stress testing, and OOD evaluation.
- Never tune hyperparameters, thresholds, calibration, or ensemble weights on `D_test` or `D_guard`.
- Every reported model must use the same `D_test` and `D_guard` samples.

Models to evaluate:
- Dummy majority baseline.
- Logistic Regression raw pixels.
- XGBoost raw pixels.
- XGBoost PCA features.
- CNN + XGBoost.
- CNN + MLP.
- CNN + MLP + classical augmentation.
- CNN + MLP + GAN augmentation.
- World Model embeddings + Logistic Regression.
- World Model embeddings + XGBoost.
- World Model embeddings + MLP.
- World Model OOD scoring.

Model combinations to evaluate:
- Probability averaging: calibrated average of two or more model probabilities.
- Weighted probability averaging: weights selected only on `D_calib`.
- Stacking: train a meta-classifier on `D_calib` predictions only, then evaluate once on `D_test` and `D_guard`.
- Feature fusion: concatenate CNN embeddings and world-model embeddings, then train XGBoost or MLP.
- Hybrid tree models: XGBoost on `[PCA features + CNN embeddings]`, `[PCA features + world-model embeddings]`, and `[CNN embeddings + world-model embeddings]`.
- Robustness ensemble: combine the best in-distribution model with the best `D_guard` model and report the trade-off.

Mandatory result tables:
- `model_level_results`: one row per model, split, calibration method, and threshold strategy.
- `combination_results`: one row per ensemble, stacking model, or feature-fusion model.
- `robustness_results`: one row per model and stress-test condition in `D_guard`.
- `efficiency_results`: fit time, inference time, parameter count when available, and memory footprint when practical.
- `error_analysis_results`: false positives, false negatives, confidence buckets, and representative image paths.

Mandatory columns:
- `model`, `model_family`, `feature_source`, `combination_type`, `split`, `calibration_method`, `threshold_strategy`, `threshold`.
- `accuracy`, `balanced_accuracy`, `precision`, `recall`, `specificity`, `f1`, `mcc`, `roc_auc`, `log_loss`, `brier_score`.
- `predicted_positive_rate`, `prevalence`, `fit_time_seconds`, `inference_time_seconds`.
- `D_test_score`, `D_guard_score`, `robustness_gap`, `ood_score` when applicable.

Supermodularity / synergy analysis:
- For each combination, compare against each component model and against the best single model.
- Compute absolute gain: `combined_metric - best_component_metric`.
- Compute expected additive gain when possible: `(model_a_metric - baseline_metric) + (model_b_metric - baseline_metric)`.
- Compute interaction score: `combined_gain - expected_additive_gain`.
- Report synergy separately for `D_test` and `D_guard`.
- A combination is useful only if it improves robustness, calibration, or interpretability without an unacceptable efficiency cost.

Ranking rule:
- Do not select the final model using accuracy alone.
- The final decision must consider `MCC`, `balanced_accuracy`, `ROC-AUC`, `log_loss`, `Brier score`, `robustness_gap`, calibration quality, SHAP/error-analysis evidence, and computational cost.
- If the best `D_test` model is weaker on `D_guard`, explain whether the robust model should be preferred.
- Keep weak models in the tables; they are part of the evidence.

Final comparison questions:
1. Which single model is best in-distribution?
2. Which single model is most robust on `D_guard`?
3. Which model is best calibrated?
4. Which combination has the strongest positive interaction score?
5. Which combination gives the best robustness/complexity trade-off?
6. Does the world-model representation add information beyond CNN embeddings?
7. Does GAN augmentation improve generalization or only training performance?
8. Which model would you ship and why?

## 14. SHAP Explainability

Goal: explain what the best-performing models are using to make decisions.

Mandatory SHAP work:
- Compute SHAP values for at least one tree-based model, for example XGBoost on raw pixels, XGBoost on PCA features, or XGBoost on CNN embeddings.
- Display a global SHAP summary plot.
- Display local SHAP explanations for at least one correctly classified image and one misclassified image.
- If the selected best model is a CNN + MLP, use SHAP image explanations or explain the CNN embeddings with SHAP through the downstream classifier.
- Compare SHAP explanations with the visual error analysis: does the model focus on triangle geometry, color, position, noise, or background artifacts?
- State whether SHAP supports the claim that the model generalizes instead of exploiting shortcuts.

Questions
1. Which pixels, PCA components, or embedding dimensions contribute most to the prediction?
2. Are the most important features geometrically meaningful?
3. Do wrong predictions show evidence of background or artifact reliance?
4. Does SHAP change the final model choice?

## 15. Optuna Optimization

Take the best model and optimize it with optuna

## 16. Create the best model you can create

After selecting the best model, perform a final manual sanity test:
- Draw one new right triangle manually on a paper.
- Draw one new non-right triangle manually on a paper.
- Save both images in the final results folder.
- Apply exactly the same preprocessing pipeline used by the selected model.
- Display both drawings in the notebook with `y_proba`, `y_pred`, and the selected threshold.
- Explain whether the model prediction matches the human geometric expectation and what this says about robustness.